In [1]:
# Auxiliary Functions
# Process and compares the dataset labels and the model output  (d is defined as dictionary and these are mutable by design)        
def iter_answer(data,table_eq,h="",l="",depth = 0):
    for n in table_eq:
        if depth == 0:
            h = n
        if isinstance(table_eq[n],dict):
            a = iter_answer(data,table_eq[n],h,n,depth+1)   
        else:
            for m in table_eq[n]:
            
                if (depth == 1):
                    if m not in data[n]:
                        continue
                    if data[n][m] > 0:
                        d[h] = 1
                elif depth == 2:
                    if data[l][n][m] > 0:
                        d[h] = 1

                
    return


In [2]:
# Reading and Defining
import json
Processed_Name = "Qwen25_3B_Presidio_ShareGPT_cap_v1.json"
dataset_name = f"./processed/{Processed_Name}"
base_dataset = "./data/base_data.json"
eqi_name = "./data/table_eq_ShareGPT.json"
data = {}
IDs = []
bdata = {}

# Loads the equivalence table between Dataset Labels and Model Output.
with open(eqi_name, 'r') as file:
    table_eqi = json.load(file)
# Open and read the model output JSON file & discards repeated conversations.


with open(dataset_name, 'r') as file:
    data_1 = json.load(file)

for i,dat in enumerate(data_1.keys()):
    if dat not in IDs:
        IDs.append(dat)
        data[dat] = data_1[dat] 
# Open and read the dataset JSON file & saves only the Labels of the ones processed by the model.
data_1 = []      
with open(base_dataset, 'r') as file:
    base_data = json.load(file)

Parsed = [i['id'] for i in base_data if i['id'] in IDs]  


for i in base_data:
    PIIs = {'DATE_TIME':0,
    'EMAIL_ADDRESS':0,
    'LOCATION':0,
    'NRP':0,
    'PASSPORT_NUMBER':0,
    'PERSON':0,
    'PHONE_NUMBER':0,
    'URL':0}
    for j,r in enumerate(PIIs):
        PIIs[r] = i['PII'][j]
    bdata[i['id']] = PIIs

# Metric Initialization variables
tp = [0]*8
tn = [0]*8
fp = [0]*8
fn = [0]*8
rec = [0]*8
prec = [0]*8
f1 = [0]*8
dd = {}

# d contains the number of the model data fields that are a match to a dataset Label.
for k in data:
    d = {'DATE_TIME':0,
    'EMAIL_ADDRESS':0,
    'LOCATION':0,
    'NRP':0,
    'PASSPORT_NUMBER':0,
    'PERSON':0,
    'PHONE_NUMBER':0,
    'URL':0}
    iter_answer (data[k],table_eqi)
    dd[k] = d
    
# Calculate the sum of all false positives (fp), true positives (tp), false negatives (fn) and true negatives (tn)
# note: dat[ID][LABEL] & base_data[ID][LABEL]. We just check if the associated data is present because the dataset may only indicate so.
for n in Parsed:
    for i,h in enumerate(bdata[n]):
        if bdata[n][h] > 0 and dd[n][h] > 0:
            tp[i] += 1
        elif bdata[n][h] == 0 and dd[n][h] == 0:
            tn[i] += 1
        elif bdata[n][h] == 0 and dd[n][h] > 0:
            fp[i] += 1
        elif bdata[n][h] > 0 and dd[n][h] == 0:
            fn[i] += 1

# Calculate the recall, precision and F1 score per label. note: If the metric results is NaN we assume the metric is 0%
for i,v in enumerate(tp):
    if ((tp[i] + fn[i]) != 0):
        rec[i] = tp[i]/(tp[i]+fn[i])
    else:
        rec[i] = 0
    if ((tp[i] +fp[i]) != 0):
        prec[i] = tp[i]/(tp[i]+fp[i])
    else:
        prec[i] = 0
    if ((rec[i] + prec[i]) != 0):
        f1[i] = 2*(rec[i]*prec[i])/(rec[i]+prec[i])
    else:
        f1[i] = 0

# Report Metrics per Label
print("['DATE_TIME', 'EMAIL', 'LOCATION', 'NRP', 'PASSPORT', 'PERSON', 'PHONE','URL'")
print(f"Prec:")
print(["%.2f" % p for p in prec] )
print(f"Recall:")
print(["%.2f" % p for p in rec] )
print(f"F1:")
print(["%.2f" % p for p in f1] )

# Calculate & report aggregated metrics
ttp = 0
ttn = 0
tfp = 0
tfn = 0
prec_m = 0
rec_m = 0
f1_m = 0
for i,ka in enumerate(tp):
    ttp += tp[i]
    ttn += tn[i]
    tfp += fp[i]
    tfn += fn[i]
    prec_m += prec[i]
    rec_m += rec[i]
    f1_m += f1[i]
    
print(f'Prec_micro: {ttp/(ttp+tfp):.2f}')
print(f'Recall_micro: {ttp/(ttp+tfn):.2f}')
print(f'F1_micro: {2*((ttp/(ttp+tfp))*(ttp/(ttp+tfn)))/((ttp/(ttp+tfp))+(ttp/(ttp+tfn))):.2f}')

print(f'Prec_macro: {prec_m/8:.2f}')
print(f'Recall_macro: {rec_m/8:.2f}')
print(f'F1_macro: {f1_m/8:.2f}')


['DATE_TIME', 'EMAIL', 'LOCATION', 'NRP', 'PASSPORT', 'PERSON', 'PHONE','URL'
Prec:
['0.62', '0.57', '0.47', '0.31', '0.33', '0.35', '0.40', '0.20']
Recall:
['0.56', '0.57', '0.52', '0.31', '1.00', '0.40', '0.29', '0.25']
F1:
['0.59', '0.57', '0.49', '0.31', '0.50', '0.37', '0.33', '0.22']
Prec_micro: 0.44
Recall_micro: 0.46
F1_micro: 0.45
Prec_macro: 0.41
Recall_macro: 0.49
F1_macro: 0.42


In [58]:
bdata[n][0]

0

In [60]:
dd[n]

{'DATE_TIME': 0,
 'EMAIL_ADDRESS': 0,
 'LOCATION': 0,
 'NRP': 0,
 'PASSPORT_NUMBER': 0,
 'PERSON': 1,
 'PHONE_NUMBER': 0,
 'URL': 0}

In [74]:
import json
with open('test.json', 'w') as f:
    json.dump(table_eqi, f, indent=4)
